# M-Smart Flow: train both models on Colab
Trains two separate YOLO26 Nano models:
1. **Parking model** (`car` / `free`) from `Parking Lot.v1i.yolo26.zip`
2. **Plate character model** from `LPR plate.v1i.yolo26.zip`

**Before you start**
- Runtime > Change runtime type > **T4 GPU**
- Put both zips in Google Drive at `MyDrive/msmartflow/` (keep the file names)
- Run the cells in order. Results are saved to Drive, so a disconnect won't lose them.

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

Write down the `ultralytics` version above and pin it in `requirements.txt` later, so the weights load the same way on your laptop.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
BASE = Path('/content/drive/MyDrive/msmartflow')
RUNS = BASE / 'runs'
MODELS = BASE / 'models'
START_WEIGHTS = MODELS / 'parking_yolo26n.pt'   # your trained v1 weights (upload this file to Drive models/)
for p in (RUNS, MODELS): p.mkdir(parents=True, exist_ok=True)

ZIPS = {
    'parking':     BASE / 'Parking Lot Combined.zip',
}
for k, z in ZIPS.items():
    print(k, 'OK' if z.exists() else 'MISSING', z)

## 1. Unzip and fix `data.yaml` paths
Unzips to Colab's local disk (faster than Drive) and rewrites the paths so training works from anywhere.

In [ ]:
import zipfile, yaml

DATA = Path('/content/datasets')
def prepare(key):
    dest = DATA / key
    if not dest.exists():
        dest.mkdir(parents=True)
        with zipfile.ZipFile(ZIPS[key]) as zf:
            zf.extractall(dest)
    yamls = list(dest.rglob('data.yaml'))
    assert yamls, f'no data.yaml found for {key}'
    root = yamls[0].parent
    d = yaml.safe_load((root / 'data.yaml').read_text())
    d['path'] = str(root)
    for k, sub in (('train', 'train'), ('val', 'valid'), ('test', 'test')):
        if (root / sub / 'images').exists():
            d[k] = f'{sub}/images'
        else:
            d.pop(k, None)
    (root / 'data.yaml').write_text(yaml.safe_dump(d, sort_keys=False))
    n = {s: len(list((root / s / 'images').glob('*'))) for s in ('train', 'valid', 'test') if (root / s / 'images').exists()}
    print(f'{key}: {len(d["names"])} classes, images per split: {n}')
    print('  classes:', d['names'])
    return root / 'data.yaml'

PARKING_YAML = prepare('parking')


Check the output above: the parking set should show **2 classes** (`car`, `free`). The plate set should show one class per character (plus the province label). If either looks wrong, stop and send me the output.

## 2. Training helper

In [ ]:
from ultralytics import YOLO

def train(data_yaml, name, base='yolo26n.pt', imgsz=640, epochs=50, batch=16):
    model = YOLO(base)  # COCO-pretrained, then fine-tuned on your data
    model.train(
        data=str(data_yaml), epochs=epochs, imgsz=imgsz, batch=batch,
        patience=20,                 # stop early if validation stops improving
        project=str(RUNS), name=name, exist_ok=True, device=0,
    )
    return RUNS / name / 'weights' / 'best.pt'

## 3. Train the parking model

In [ ]:
parking_best = train(PARKING_YAML, 'parking_v2', base=str(START_WEIGHTS))   # fine-tune from v1 weights, saved as runs/parking_v2
print(parking_best)

## 5. Evaluate on the held-out **test** split
Note: the parking dataset's own test split likely shares scenes with train (augmented copies), so treat this as an *in-distribution* score. Your real test is your own unseen mall clips, run locally later.

In [ ]:
m = YOLO(str(parking_best))
r = m.val(data=str(PARKING_YAML), split='test', device=0)
print(f'== parking_v2: mAP50={r.box.map50:.3f}  mAP50-95={r.box.map:.3f}  P={r.box.mp:.3f}  R={r.box.mr:.3f}')

## 6. Copy the final weights to Drive `models/`

In [ ]:
import shutil
shutil.copy(parking_best, MODELS / 'parking_v2.pt')
print(sorted(p.name for p in MODELS.iterdir()))

Download the two `.pt` files from `MyDrive/msmartflow/models/` into the project's `models/` folder. Also keep `runs/<name>/results.csv` and `results.png` for the senior-project report.

**If Colab disconnects mid-training:** reconnect, re-run the setup cells (up to the helper), then resume with
`YOLO(str(RUNS/'parking_yolo26n'/'weights'/'last.pt')).train(resume=True)`